# Bài tập Spider Chart (Radar Chart)
### Tập dữ liệu: Salaries.csv

## 1. Import thư viện và đọc dữ liệu

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('dataset/Salaries.csv')
print(df.shape)
df.head()

(78, 6)


,rank,discipline,phd,service,sex,salary
0,Prof,B,56,49,Male,186960
1,Prof,A,12,6,Male,93000
2,Prof,A,23,20,Male,110515
3,Prof,A,40,31,Male,131205
4,Prof,B,20,18,Male,104800


---
## 2. Spider Chart so sánh theo Rank
**Categories (trục):** Discipline A và B × 3 chỉ số: Salary, PhD years, Service years  
**Thể loại so sánh:** `rank` (Prof / AssocProf / AsstProf)  

> **Lý do dùng 6 trục:** Spider chart cần ít nhất 3 trục để tạo thành hình đa giác.  
> Chỉ có 2 discipline (A, B) nên ta kết hợp discipline × metric để có 6 trục rõ ràng.

In [ ]:

g = df.groupby(['rank','discipline'])[['salary','phd','service']].mean().round(1)

categories = ['Salary_A', 'Salary_B', 'PhD_A', 'PhD_B', 'Service_A', 'Service_B']

def normalize(vals):
    mn, mx = min(vals), max(vals)
    return [round((v - mn) / (mx - mn) * 100, 1) for v in vals]

ranks = ['Prof', 'AssocProf', 'AsstProf']
raw = {}
for r in ranks:
    raw[r] = [
        g.loc[(r,'A'), 'salary'],  g.loc[(r,'B'), 'salary'],
        g.loc[(r,'A'), 'phd'],     g.loc[(r,'B'), 'phd'],
        g.loc[(r,'A'), 'service'], g.loc[(r,'B'), 'service'],
    ]


for i, name in enumerate(['salary','phd','service']):
    col_vals = [raw[r][i*2] for r in ranks] + [raw[r][i*2+1] for r in ranks]
    mn, mx = min(col_vals), max(col_vals)
    for r in ranks:
        for j in range(2):
            raw[r][i*2+j] = round((raw[r][i*2+j]-mn)/(mx-mn)*100, 1)


rows = []
for r in ranks:
    vals = raw[r]
    for c, v in zip(categories, vals):
        rows.append({'rank': r, 'category': c, 'value': v})
df_plot1 = pd.DataFrame(rows)
print(df_plot1)

         rank   category  value
0        Prof   Salary_A   61.9
1        Prof   Salary_B  100.0
2        Prof      PhD_A  100.0
3        Prof      PhD_B   91.1
4        Prof  Service_A   90.1
5        Prof  Service_B  100.0
6   AssocProf   Salary_A    0.0
7   AssocProf   Salary_B   47.2
8   AssocProf      PhD_A   60.8
9   AssocProf      PhD_B   35.4
10  AssocProf  Service_A   67.5
11  AssocProf  Service_B   31.5
12   AsstProf   Salary_A    0.8
13   AsstProf   Salary_B   20.3
14   AsstProf      PhD_A    0.0
15   AsstProf      PhD_B    4.6
16   AsstProf  Service_A    0.0
17   AsstProf  Service_B    1.0


In [3]:
fig1 = px.line_polar(
    df_plot1,
    r='value',
    theta='category',
    color='rank',
    line_close=True,
    markers=True,
    title='Spider Chart: So sánh theo Rank và Discipline<br><sup>(Giá trị chuẩn hóa 0–100 theo từng chỉ số)</sup>',
    color_discrete_map={
        'Prof':      '#2196F3',
        'AssocProf': '#FF9800',
        'AsstProf':  '#4CAF50'
    }
)

fig1.update_traces(fill='toself', opacity=0.45)

fig1.update_layout(
    title_x=0.5,
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 100],
                        tickvals=[0,25,50,75,100]),
        angularaxis=dict(tickfont=dict(size=12))
    ),
    legend=dict(title='Rank', orientation='h',
                xanchor='center', x=0.5, yanchor='bottom', y=-0.22),
    margin=dict(l=60, r=60, t=100, b=100),
    width=650, height=600
)

fig1.show()

### Nhận xét – Spider Chart theo Rank

**Quan sát từ biểu đồ:**
- **Prof** (xanh dương) có diện tích phủ **lớn nhất** — vượt trội trên hầu hết 6 trục, đặc biệt ở `Salary_B` (~135,314 USD) và `Service_B` (~22 năm kinh nghiệm).
- **AssocProf** (cam) nằm ở vị trí **trung gian** — lương và kinh nghiệm đều thấp hơn Prof nhưng cao hơn AsstProf.
- **AsstProf** (xanh lá) có diện tích **nhỏ nhất** — phản ánh giai đoạn đầu sự nghiệp: ít kinh nghiệm, lương thấp nhất ở cả hai ngành.
- Trên cả 3 cấp bậc, **ngành B** luôn có lương cao hơn ngành A (trục `Salary_B` > `Salary_A`).
- **PhD years và Service years** cũng tăng theo cấp bậc — Prof có thâm niên học thuật lâu nhất.

**Kết luận:** Biểu đồ xác nhận mối quan hệ tuyến tính rõ ràng giữa cấp bậc học thuật và tất cả các chỉ số. Ngành B (Applied) trả lương cao hơn ngành A (Theoretical) ở mọi cấp bậc.

---
## 3. Spider Chart so sánh theo Sex
**Categories (trục):** Discipline A và B × 3 chỉ số: Salary, PhD years, Service years  
**Thể loại so sánh:** `sex` (Male / Female)  
*(Dùng `go.Scatterpolar` theo slide bài học)*

In [4]:
g2 = df.groupby(['sex','discipline'])[['salary','phd','service']].mean().round(1)

sexes = ['Male', 'Female']
raw2 = {}
for s in sexes:
    raw2[s] = [
        g2.loc[(s,'A'), 'salary'],  g2.loc[(s,'B'), 'salary'],
        g2.loc[(s,'A'), 'phd'],     g2.loc[(s,'B'), 'phd'],
        g2.loc[(s,'A'), 'service'], g2.loc[(s,'B'), 'service'],
    ]


for i in range(3):
    col_vals = [raw2[s][i*2] for s in sexes] + [raw2[s][i*2+1] for s in sexes]
    mn, mx = min(col_vals), max(col_vals)
    for s in sexes:
        for j in range(2):
            raw2[s][i*2+j] = round((raw2[s][i*2+j]-mn)/(mx-mn)*100, 1)

print("Male values (normalized):",   raw2['Male'])
print("Female values (normalized):", raw2['Female'])

Male values (normalized): [np.float64(57.3), np.float64(100.0), np.float64(100.0), np.float64(49.5), np.float64(100.0), np.float64(68.6)]
Female values (normalized): [np.float64(0.0), np.float64(68.5), np.float64(18.2), np.float64(0.0), np.float64(0.0), np.float64(3.5)]


In [ ]:
cats = categories + [categories[0]]  

fig2 = go.Figure()

colors = {'Male': ('steelblue', 'rgba(70,130,180,0.35)'),
          'Female': ('tomato', 'rgba(255,99,71,0.35)')}

for s in sexes:
    vals = raw2[s] + [raw2[s][0]]  
    fig2.add_trace(go.Scatterpolar(
        r=vals,
        theta=cats,
        fill='toself',
        name=s,
        line=dict(color=colors[s][0], width=2),
        fillcolor=colors[s][1]
    ))

fig2.update_layout(
    title='Spider Chart: So sánh theo Sex và Discipline<br><sup>(Giá trị chuẩn hóa 0–100 theo từng chỉ số)</sup>',
    title_x=0.5,
    polar=dict(
        radialaxis=dict(visible=True, range=[0,100],
                        tickvals=[0,25,50,75,100]),
        angularaxis=dict(tickfont=dict(size=12))
    ),
    legend=dict(title='Sex', orientation='h',
                xanchor='center', x=0.5, yanchor='bottom', y=-0.2),
    margin=dict(l=60, r=60, t=100, b=100),
    width=650, height=600,
    showlegend=True
)

fig2.show()

### Nhận xét – Spider Chart theo Sex

**Quan sát từ biểu đồ:**
- **Male** (xanh) có diện tích phủ **lớn hơn Female** trên hầu hết các trục, thể hiện mức lương và kinh nghiệm cao hơn.
- Chênh lệch lương rõ nhất ở **ngành A**: Male ~107,597 USD so với Female ~89,065 USD (chênh ~21%).
- Ở **ngành B** chênh lệch nhỏ hơn: Male ~121,429 USD vs Female ~111,235 USD (~9%).
- **PhD years và Service years**: Male cũng cao hơn Female ở ngành A (thâm niên lâu hơn đáng kể), trong khi ngành B gần tương đương.
- Hình dạng hai đa giác **tương đồng** nhưng Male luôn xa tâm hơn — cho thấy chênh lệch có tính hệ thống.

**Kết luận:** Có sự chênh lệch lương theo giới tính, rõ nhất ở ngành A. Tuy nhiên, khoảng cách về `service` và `phd` gợi ý rằng một phần chênh lệch lương có thể do Male có kinh nghiệm nhiều hơn, đặc biệt ở ngành A. Cần phân tích thêm với kiểm soát biến rank để kết luận về bất bình đẳng giới thực sự.